# ✈️ Aviation VQA — CNN Only (ResNet50 Pixel Reading)
No CLIP. No ChromaDB. Pure CNN pixel reading → answer.

> **Before starting:** Runtime → Change runtime type → **T4 GPU**

## Cell 0 — Run First Every Session

In [ ]:
# ════════════════════════════════════════════════
# CELL 0 — RUN THIS FIRST EVERY SESSION
# ════════════════════════════════════════════════
import os, sys, zipfile, json, shutil

# 1. Mount Drive
from google.colab import drive
drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/aviation_vqa_output"
PROC_DIR = os.path.join(BASE_DIR, "data/processed")
RAW_DIR  = os.path.join(BASE_DIR, "data/raw")
print(f"BASE_DIR: {BASE_DIR}")

# 2. Remove old broken extraction
if os.path.exists("/content/aviation_cnn"):
    shutil.rmtree("/content/aviation_cnn")

# 3. Find and extract zip
zip_locations = [
    "/content/aviation_cnn.zip",
    os.path.join(BASE_DIR, "aviation_cnn.zip"),
    "/content/drive/MyDrive/aviation_cnn.zip",
]
found = None
for p in zip_locations:
    if os.path.exists(p):
        found = p
        break

if found:
    print(f"Found zip: {found}")
    with zipfile.ZipFile(found, "r") as z:
        z.extractall("/content/")
    print("Extracted ✓")
else:
    from google.colab import files
    print("Upload aviation_cnn.zip now:")
    uploaded = files.upload()
    for fname in uploaded:
        with zipfile.ZipFile(fname, "r") as z:
            z.extractall("/content/")
    # Save to Drive for next session
    for fname in uploaded:
        dst = os.path.join(BASE_DIR, "aviation_cnn.zip")
        if not os.path.exists(dst):
            shutil.copy(fname, dst)
            print(f"Zip saved to Drive ✓")
    print("Extracted ✓")

# 4. Fix __init__.py
MODULES = ["data_collection","radar_generation","qa_generation",
           "preprocessing","models","evaluation","voice","gui"]
for mod in MODULES:
    path = f"/content/aviation_cnn/{mod}/__init__.py"
    if not os.path.exists(path):
        with open(path, "w") as f: f.write(f"# {mod}\n")

# 5. Clean sys.path
sys.path = [p for p in sys.path if "/content/aviation_cnn" not in p]
sys.path.insert(0, "/content/aviation_cnn")

# 6. Load raw_data if exists
raw_manifest = os.path.join(RAW_DIR, "raw_frames.json")
if os.path.exists(raw_manifest):
    with open(raw_manifest) as f:
        raw_data = json.load(f)
    print(f"raw_data: {len(raw_data)} frames ✓")
else:
    raw_data = []
    print("raw_data not found — Step 3 will generate it")

# 7. Test imports
print("\nTesting imports...")
tests = [
    ("data_collection.opensky_collector","SyntheticOpenSkyCollector"),
    ("radar_generation.radar_renderer","RadarRenderer"),
    ("qa_generation.qa_generator","RadarQAGenerator"),
    ("preprocessing.dataset_builder","DatasetBuilder"),
    ("models.cnn_model","CNNVQAModel"),
    ("models.cnn_dataset","CNNVQADataset"),
    ("models.cnn_trainer","CNNVQATrainer"),
    ("models.cnn_inference","CNNInferenceEngine"),
    ("evaluation.evaluator","CNNEvaluator"),
    ("voice.voice_query","TextQueryInterface"),
    ("gui.cnn_demo","CNNPilotDemo"),
]
ok, fail = [], []
for module, cls in tests:
    try:
        mod = __import__(module, fromlist=[cls])
        getattr(mod, cls)
        ok.append(module)
        print(f"  ✓ {module}")
    except Exception as e:
        fail.append(module)
        print(f"  ✗ {module} — {e}")

print(f"\n{len(ok)}/{len(tests)} imports OK")
if not fail:
    print("All good ✓ Proceed to Step 1")
else:
    print("Fix failed imports before continuing")


## Step 1 — Install Packages

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip"] + list(args),
                          stdout=subprocess.DEVNULL,
                          stderr=subprocess.DEVNULL)

print("Installing packages...")
pip("install", "-q",
    "torch", "torchvision", "requests",
    "pillow", "matplotlib", "numpy", "pandas",
    "scikit-learn", "sounddevice",
    "SpeechRecognition", "scipy",
    "tqdm", "ipywidgets")

import torch
print(f"PyTorch : {torch.__version__} ✓")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
print("Packages ready ✓")


## Step 2 — Setup Directories

In [ ]:
import os, json, torch

DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
PROC_DIR = os.path.join(BASE_DIR, "data/processed")
RAW_DIR  = os.path.join(BASE_DIR, "data/raw")
IMG_DIR  = os.path.join(BASE_DIR, "data/raw/images")

for d in ["data/raw/images","data/processed/images",
          "data/annotations","models",
          "evaluation_results","demo_sessions"]:
    os.makedirs(os.path.join(BASE_DIR, d), exist_ok=True)

print(f"Device   : {DEVICE}")
print(f"BASE_DIR : {BASE_DIR}")
print("Directories ready ✓")


## Step 3 — Collect 2000 NEW Frames (Total = 4000, No Duplicates)

In [ ]:
import os, sys, json, hashlib
sys.path.insert(0, "/content/aviation_cnn")

RAW_DIR      = os.path.join(BASE_DIR, "data/raw")
OLD_MANIFEST = os.path.join(RAW_DIR, "raw_frames.json")
NEW_MANIFEST = os.path.join(RAW_DIR, "all_frames.json")

# Load existing 2000 frames
if os.path.exists(OLD_MANIFEST):
    with open(OLD_MANIFEST) as f:
        existing_frames = json.load(f)
    print(f"Existing frames: {len(existing_frames)}")
else:
    existing_frames = []
    print("No existing frames found")

# Check if we already have 4000
if os.path.exists(NEW_MANIFEST):
    with open(NEW_MANIFEST) as f:
        all_frames = json.load(f)
    print(f"Loaded {len(all_frames)} total frames from Drive ✓")
    raw_data = all_frames
else:
    start_idx = len(existing_frames)
    print(f"Collecting 2000 NEW frames (starting at frame_{start_idx:04d}) ...")

    USE_SYNTHETIC = False
    if not USE_SYNTHETIC:
        try:
            import requests
            r = requests.get(
                "https://opensky-network.org/api/states/all",
                params={"lamin":36,"lomin":-10,"lamax":60,"lomax":25},
                timeout=10)
            r.raise_for_status()
            from data_collection.opensky_collector import OpenSkyCollector
            collector = OpenSkyCollector(
                output_dir=RAW_DIR,
                num_new_frames=2000,
                existing_frames=existing_frames,
                aircraft_per_frame=(4,10),
                sleep_between=2.0)
            print("Using OpenSky API ...")
        except Exception as e:
            print(f"OpenSky unavailable ({e}). Using synthetic.")
            USE_SYNTHETIC = True

    if USE_SYNTHETIC:
        from data_collection.opensky_collector import SyntheticOpenSkyCollector
        collector = SyntheticOpenSkyCollector(
            output_dir=RAW_DIR,
            num_new_frames=2000,
            existing_frames=existing_frames,
            aircraft_per_frame=(4,10),
            sleep_between=0.0)

    new_frames = collector.collect(start_idx=start_idx)
    all_frames = existing_frames + new_frames

    # Verify NO duplicates across all 4000
    fps = set()
    dups = 0
    for fr in all_frames:
        key = ",".join(sorted(
            f"{a['latitude']:.2f},{a['longitude']:.2f}"
            for a in fr["aircraft"]))
        fp = hashlib.md5(key.encode()).hexdigest()
        if fp in fps: dups += 1
        fps.add(fp)

    print(f"\nTotal frames : {len(all_frames)}")
    print(f"Duplicates   : {dups} (should be 0)")
    assert dups == 0, "Duplicates found!"

    with open(NEW_MANIFEST, "w") as f:
        json.dump(all_frames, f, indent=2)
    print(f"Saved {len(all_frames)} frames ✓")
    raw_data = all_frames


## Step 4 — Render 2000 NEW Radar Images (Total = 4000 images)

In [ ]:
import os, sys, json
sys.path.insert(0, "/content/aviation_cnn")
from radar_generation.radar_renderer import RadarRenderer
from PIL import Image
import matplotlib.pyplot as plt, random

IMG_DIR  = os.path.join(BASE_DIR, "data/raw/images")
REN_MAN  = os.path.join(BASE_DIR, "data/raw/rendered_frames.json")

renderer = RadarRenderer(output_dir=IMG_DIR, image_size=224)

if os.path.exists(REN_MAN):
    with open(REN_MAN) as f:
        rendered = json.load(f)
    # Check for new frames not yet rendered
    rendered_ids = {fr["frame_id"] for fr in rendered}
    new_to_render = [fr for fr in raw_data
                     if fr["frame_id"] not in rendered_ids]
    if new_to_render:
        print(f"Rendering {len(new_to_render)} new frames...")
        # Attach existing paths first
        renderer.attach_paths(rendered, IMG_DIR)
        new_rendered = renderer.render_all(new_to_render)
        rendered = rendered + new_rendered
        with open(REN_MAN, "w") as f:
            json.dump(rendered, f, indent=2)
        print(f"Total rendered: {len(rendered)} ✓")
    else:
        print(f"All {len(rendered)} frames already rendered ✓")
else:
    print(f"Rendering all {len(raw_data)} frames...")
    # Attach existing raw images if already there
    existing_rendered = renderer.attach_paths(
        [fr for fr in raw_data
         if os.path.exists(
             os.path.join(IMG_DIR, f"{fr['frame_id']}.png"))],
        IMG_DIR)
    rendered_ids = {fr["frame_id"] for fr in existing_rendered}
    to_render = [fr for fr in raw_data
                 if fr["frame_id"] not in rendered_ids]
    new_rendered = renderer.render_all(to_render)
    rendered = existing_rendered + new_rendered
    with open(REN_MAN, "w") as f:
        json.dump(rendered, f, indent=2)
    print(f"Rendered {len(rendered)} images ✓")

# Preview
samples = random.sample(rendered, min(4, len(rendered)))
fig, axes = plt.subplots(1, len(samples),
                          figsize=(4*len(samples), 4))
if len(samples)==1: axes=[axes]
for ax, fr in zip(axes, samples):
    ax.imshow(Image.open(fr["image_path"]))
    ax.set_title(f"{fr['frame_id']}\n"
                 f"{len(fr['aircraft'])} aircraft", fontsize=8)
    ax.axis("off")
plt.suptitle(f"Sample Radar Images (Total: {len(rendered)})")
plt.tight_layout(); plt.show()


## Step 5 — Generate QA Pairs (~28,000 total for 4000 images)

In [ ]:
import os, sys, json
sys.path.insert(0, "/content/aviation_cnn")
from qa_generation.qa_generator import RadarQAGenerator
from collections import Counter

ANNO_PATH = os.path.join(BASE_DIR,
    "data/annotations/annotations.jsonl")

if os.path.exists(ANNO_PATH):
    with open(ANNO_PATH) as f:
        annotations = [json.loads(l) for l in f if l.strip()]
    print(f"Loaded {len(annotations)} existing QA pairs")

    # Check if new frames need QA
    existing_image_ids = {r["image_id"] for r in annotations}
    new_frames_for_qa = [
        fr for fr in rendered
        if fr["frame_id"] not in existing_image_ids]

    if new_frames_for_qa:
        print(f"Generating QA for {len(new_frames_for_qa)} new frames...")
        qa_gen = RadarQAGenerator()
        new_qa = qa_gen.generate_all(
            frames=new_frames_for_qa,
            qa_per_image=7,
            output_path=ANNO_PATH,
            append=True)
        annotations = annotations + new_qa
        print(f"Total QA pairs: {len(annotations)} ✓")
    else:
        print("All frames have QA pairs ✓")
else:
    print(f"Generating QA for {len(rendered)} frames...")
    for fr in rendered:
        if "image_path" not in fr:
            fr["image_path"] = os.path.join(
                IMG_DIR, f"{fr['frame_id']}.png")
    qa_gen = RadarQAGenerator()
    annotations = qa_gen.generate_all(
        frames=rendered, qa_per_image=7,
        output_path=ANNO_PATH)
    print(f"Generated {len(annotations)} QA pairs ✓")

dist = Counter(r["question_type"] for r in annotations)
print(f"\nTotal: {len(annotations):,} QA pairs")
print("\nQuestion type distribution:")
for t, c in sorted(dist.items(), key=lambda x:-x[1]):
    print(f"  {t:<30} {c:>6}")


## Step 6 — Preprocess & 70/30 Train/Test Split

In [ ]:
import os, sys, json
sys.path.insert(0, "/content/aviation_cnn")
from preprocessing.dataset_builder import DatasetBuilder

PROC_DIR = os.path.join(BASE_DIR, "data/processed")
ANNO_PATH= os.path.join(BASE_DIR,
    "data/annotations/annotations.jsonl")
TRAIN_J  = os.path.join(PROC_DIR, "train.json")

if os.path.exists(TRAIN_J):
    with open(os.path.join(PROC_DIR,"answer_index.json")) as f:
        answer_index = json.load(f)
    with open(TRAIN_J) as f: tr = json.load(f)
    with open(os.path.join(PROC_DIR,"test.json")) as f:
        te = json.load(f)
    print(f"Loaded: train={len(tr):,}, "
          f"test={len(te):,}, "
          f"classes={len(answer_index)}")
else:
    builder = DatasetBuilder(
        raw_image_dir=IMG_DIR,
        annotation_path=ANNO_PATH,
        processed_dir=PROC_DIR,
        train_ratio=0.7)
    stats = builder.build()
    print(f"Train   : {stats['train']:,}")
    print(f"Test    : {stats['test']:,}")
    print(f"Classes : {stats['num_classes']}")
    print(f"Top 5   : {[a for a,_ in stats['top_answers'][:5]]}")
    with open(os.path.join(PROC_DIR,"answer_index.json")) as f:
        answer_index = json.load(f)

print(f"\nAnswer vocab: {len(answer_index)} classes ✓")


## Step 7 — Train CNN VQA Model (Target: 70%+ Accuracy)

In [ ]:
import os, sys, json, torch
sys.path.insert(0, "/content/aviation_cnn")
from models.cnn_model import CNNVQAModel
from models.cnn_dataset import CNNVQADataset
from models.cnn_trainer import CNNVQATrainer
from torch.utils.data import DataLoader

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_DIR  = os.path.join(BASE_DIR, "models")
BEST_MODEL = os.path.join(MODEL_DIR, "cnn_best_model.pth")
PROC_DIR   = os.path.join(BASE_DIR, "data/processed")

with open(os.path.join(PROC_DIR,"answer_index.json")) as f:
    answer_index = json.load(f)

if os.path.exists(BEST_MODEL):
    print("Loading trained model from Drive ...")
    model = CNNVQAModel(
        num_classes=len(answer_index), device=DEVICE)
    model.load(BEST_MODEL)
    print(f"Model loaded ✓  ({len(answer_index)} classes)")
else:
    nw = 2 if DEVICE == "cuda" else 0
    train_ds = CNNVQADataset(
        os.path.join(PROC_DIR,"train.json"), split="train")
    test_ds  = CNNVQADataset(
        os.path.join(PROC_DIR,"test.json"),  split="test")
    train_loader = DataLoader(
        train_ds, batch_size=32, shuffle=True,
        num_workers=nw, pin_memory=(DEVICE=="cuda"))
    test_loader  = DataLoader(
        test_ds,  batch_size=32, shuffle=False,
        num_workers=nw, pin_memory=(DEVICE=="cuda"))

    print(f"Train batches : {len(train_loader)}")
    print(f"Test  batches : {len(test_loader)}")
    print(f"Classes       : {len(answer_index)}")
    print(f"Device        : {DEVICE}")

    model = CNNVQAModel(
        num_classes=len(answer_index),
        device=DEVICE, feat_dim=512)

    trainable = sum(
        p.numel() for p in model.parameters()
        if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable params: {trainable:,} / {total:,}")

    trainer = CNNVQATrainer(
        model=model,
        train_loader=train_loader,
        val_loader=test_loader,
        device=DEVICE,
        epochs=20,
        save_dir=MODEL_DIR)

    history = trainer.train()
    best    = max(history["val_acc"])
    print(f"\nBest Val Accuracy: {best:.2f}%")
    if best >= 70:
        print("  ★ 70% target achieved!")
    else:
        print("  Model trained. Run more epochs if needed.")
    trainer.plot_history(history)


## Step 8 — Load CNN Inference Engine

In [ ]:
import os, sys, torch
sys.path.insert(0, "/content/aviation_cnn")
from models.cnn_inference import CNNInferenceEngine

DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
PROC_DIR  = os.path.join(BASE_DIR, "data/processed")
BEST_MODEL= os.path.join(BASE_DIR, "models/cnn_best_model.pth")

engine = CNNInferenceEngine(
    model_path        = BEST_MODEL,
    answer_index_path = os.path.join(PROC_DIR,"answer_index.json"),
    device            = DEVICE)

print("CNN Inference Engine loaded ✓")
print("Pipeline: raw pixels → ResNet50 → answer")
print("No CLIP. No ChromaDB. No embeddings.")


## Step 9 — Test Inference on Sample Image

In [ ]:
import os, random
from PIL import Image
import matplotlib.pyplot as plt

PROC_DIR  = os.path.join(BASE_DIR, "data/processed")
imgs      = os.listdir(os.path.join(PROC_DIR,"images"))
sample    = os.path.join(PROC_DIR,"images", random.choice(imgs))
img       = Image.open(sample)

plt.figure(figsize=(4,4))
plt.imshow(img)
plt.title(os.path.basename(sample))
plt.axis("off"); plt.show()

questions = [
    "How many aircraft are visible on the radar?",
    "Is there any aircraft on the left side of the radar?",
    "Is there any aircraft in the upper portion of the radar?",
    "Are there more than 4 aircraft on the radar?",
    "What is the callsign of the aircraft with the highest altitude?",
    "Which quadrant of the radar has the most aircraft?",
    "How many aircraft are flying above 30,000 feet?",
    "Is there any aircraft near the center of the radar?",
]

print(f"{'QUESTION':<55} {'ANSWER':<20} CONF")
print("-"*85)
for q in questions:
    r = engine.answer(img, q)
    print(f"{q:<55} {r['answer']:<20} {r['confidence']:.3f}")


## Step 10 — Evaluation

In [ ]:
import os, sys, json
sys.path.insert(0, "/content/aviation_cnn")
from evaluation.evaluator import CNNEvaluator

PROC_DIR = os.path.join(BASE_DIR, "data/processed")
EVAL_DIR = os.path.join(BASE_DIR, "evaluation_results")

with open(os.path.join(PROC_DIR,"answer_index.json")) as f:
    answer_index = json.load(f)

evaluator = CNNEvaluator(
    engine       = engine,
    test_json    = os.path.join(PROC_DIR,"test.json"),
    answer_index = answer_index,
    output_dir   = EVAL_DIR)

metrics = evaluator.run()
print("\n========== CNN EVALUATION RESULTS ==========")
print(f"  Overall Accuracy : {metrics['overall_acc']:>6.2f}%")
print(f"  Macro F1 Score   : {metrics['macro_f1']:>6.4f}")
print("=============================================")


In [ ]:
evaluator.plot_confusion_matrix(metrics)
evaluator.print_report(metrics)


## Step 11 — Live CNN Pilot Demo GUI

In [ ]:
import os, sys
sys.path.insert(0, "/content/aviation_cnn")
from gui.cnn_demo import CNNPilotDemo

IMAGE_DIR = os.path.join(BASE_DIR, "data/processed/images")

demo = CNNPilotDemo(
    engine    = engine,
    image_dir = IMAGE_DIR,
    save_dir  = os.path.join(BASE_DIR, "demo_sessions"))

demo.show()


## Step 12 — Voice / Text Query

In [ ]:
import os, sys, random
sys.path.insert(0, "/content/aviation_cnn")
from voice.voice_query import TextQueryInterface

PROC_DIR  = os.path.join(BASE_DIR,"data/processed")
imgs      = os.listdir(os.path.join(PROC_DIR,"images"))
img_path  = os.path.join(PROC_DIR,"images", random.choice(imgs))

ti = TextQueryInterface(engine=engine)
for q in ["How many aircraft are on the radar?",
          "Is there an aircraft near the center?",
          "Which quadrant has the most aircraft?"]:
    r = ti.query(img_path, q)
    print(f"Q: {q}\n   A: {r['answer']}  "
          f"[conf={r['confidence']:.3f}]\n")


In [ ]:
try:
    from voice.voice_query import VoiceQueryInterface
    voice = VoiceQueryInterface(engine=engine)
    print("Speak now (4 seconds) ...")
    result = voice.query_once(image_path=img_path, duration=4)
    print(f"Heard  : {result['transcript']}")
    print(f"Answer : {result['answer']}")
except Exception as e:
    print(f"Voice unavailable ({e})")
    print("Use TextQueryInterface or GUI above.")
